# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (FAIR^2 Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the Croissant metadata to enumerate all record sets and their fields by `@id` for reference. These `@id`s are required for extraction and processing steps.

In [ ]:
# List all record sets and their fields by @id
croissant_metadata = dataset.metadata.to_json()

record_sets = croissant_metadata.get('recordSet', [])

if not record_sets:
    print("No record sets found in Croissant metadata.")
else:
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) else rs
        # Retrieve record set definition by @id
        # This step assumes all referenced record sets are included in the top-level objects
        rs_obj = next((x for x in croissant_metadata.get('recordSet', []) if isinstance(x, dict) and x.get('@id') == rs_id), None)
        print(f'RecordSet: {rs_id}')
        if rs_obj and 'field' in rs_obj:
            fields = rs_obj['field'] if isinstance(rs_obj['field'], list) else [rs_obj['field']]
            for field in fields:
                if isinstance(field, dict):
                    print(f"  Field: {field.get('@id')}, name: {field.get('name')}")
                else:
                    print(f"  Field: {field}")
        else:
            print("  No fields found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Here we demonstrate extraction from the main protein record set, which is typically the largest (and only) data table. If the record sets list is empty, you may need to refer to the Croissant metadata directly to find record set `@id`s.

In [ ]:
# If recordSets are empty, identify from Croissant top-level metadata by scanning for RecordSet objects
croissant_json = dataset.metadata.to_json()
def find_record_sets(md):
    found = []
    if isinstance(md, dict):
        if md.get('@type') == 'cr:RecordSet' or md.get('@type') == 'RecordSet':
            found.append(md['@id'])
        for v in md.values():
            found.extend(find_record_sets(v))
    elif isinstance(md, list):
        for item in md:
            found.extend(find_record_sets(item))
    return found

record_set_ids = find_record_sets(croissant_json)

if record_set_ids:
    print(f"RecordSets found: {record_set_ids}")
else:
    print("No RecordSets found in metadata.")

# We'll use the main record set for protein data
main_record_set = record_set_ids[0] if record_set_ids else None

dataframes = {}
if main_record_set:
    records = list(dataset.records(record_set=main_record_set))
    df = pd.DataFrame(records)
    dataframes[main_record_set] = df
    print(f"Fields/columns in record set {main_record_set}:\n", df.columns.tolist())
    display(df.head())
else:
    print("No main record set identified; cannot extract records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates these operations using one numeric field and one group/categorical field from the extracted DataFrame.

Below, we identify likely numeric and categorical fields to use. Please adjust the field `@id`s below if your dataset uses different column names.

In [ ]:
# Preview column names
if main_record_set:
    cols = dataframes[main_record_set].columns.tolist()
    print("Columns detected:", cols)

    # Heuristically choose the first numeric/float field for demonstration
    import numpy as np
    numeric_field = None
    for col in cols:
        if pd.api.types.is_numeric_dtype(dataframes[main_record_set][col]):
            numeric_field = col
            break
    if numeric_field is None:  # Try to coerce
        for col in cols:
            try:
                dataframes[main_record_set][col] = pd.to_numeric(dataframes[main_record_set][col])
                numeric_field = col
                break
            except:
                continue

    print(f"Numeric field chosen: {numeric_field}")

    # Choose a categorical field
    group_field = None
    for col in cols:
        if pd.api.types.is_object_dtype(dataframes[main_record_set][col]):
            group_field = col
            break
    print(f"Categorical/group field chosen: {group_field}")

    if numeric_field:
        # Filter records based on a threshold (using 25th percentile)
        threshold = dataframes[main_record_set][numeric_field].quantile(0.25)
        filtered_df = dataframes[main_record_set][dataframes[main_record_set][numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group and aggregate
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
else:
    print("No data loaded. Please ensure a valid record set was found in Croissant metadata.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we create histograms and boxplots for the selected numeric field, as well as a barplot grouped by the categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_record_set and numeric_field:
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    sns.histplot(dataframes[main_record_set][numeric_field].dropna(), ax=axs[0], kde=True)
    axs[0].set_title(f"Distribution of {numeric_field}")
    sns.boxplot(x=dataframes[main_record_set][numeric_field].dropna(), ax=axs[1])
    axs[1].set_title(f"Boxplot of {numeric_field}")
    plt.tight_layout()
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.barplot(data=filtered_df, x=group_field, y=numeric_field, ci=None)
        plt.title(f"Mean {numeric_field} by {group_field} (filtered)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("Data not available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to explore a mass spectrometry dataset using the `mlcroissant` library.

**Key observations:**
- The FAIR^2 Croissant schema enables granular data exploration by providing structured `@id`-based navigation of fields and tables.
- Data extraction, filtering, and normalization steps are straightforward with `mlcroissant` and pandas.
- The dataset contains protein-centric metrics suitable for downstream proteomics analysis, filtering, and visualization.

You can extend this notebook by performing domain-specific analysis on protein abundances, searching for post-translational modifications, or integrating other FAIR-compliant datasets.